# ModernBERT-large LoRA Offline Full Submission

This notebook is generated from `src/generate_submission_modernbert_lora.py` for the Kaggle LLM Classification Finetuning competition.

It is designed for an offline Kaggle run: attach the competition data, attach ModernBERT-large as a Kaggle Model, and attach a dataset containing compatible offline wheels for PEFT, Accelerate, Transformers, and their dependencies.

The notebook trains grouped cross-validation LoRA adapters, validates with A/B swap test-time augmentation, saves fold artifacts, writes `oof_predictions.csv`, and creates the required `submission.csv`.


In [ ]:
r"""
Experiment: ModernBERT-large + LoRA + Grouped CV + A/B Swap + Swap TTA
----------------------------------------------------------------------
Full Kaggle training script for the LLM Classification Finetuning
competition. It trains one LoRA adapter per grouped fold, validates with
swap test-time augmentation, saves fold adapters, and writes submission.csv.

Recommended full run:
    .\submit_to_kaggle.ps1 `
        -ScriptFile ".\src\generate_submission_modernbert_lora.py" `
        -Message "ModernBERT-large LoRA grouped CV full run" `
        -EnableGpu `
        -Model "answer-ai/modernbert/Transformers/large/2" `
        -Dataset "your-username/your-offline-wheels-dataset"

For an offline Kaggle run, attach:
    1. the competition data
    2. ModernBERT-large weights
    3. a wheel dataset containing compatible peft/accelerate/transformers wheels
"""

from __future__ import annotations

import gc
import importlib.metadata as importlib_metadata
import json
import math
import os
import random
import re
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path


# ---------------------------------------------------------------------------


## Offline Dependency Bootstrap


In [ ]:
# ---------------------------------------------------------------------------

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MIN_DEPENDENCIES = {
    "transformers": "4.52.1",
    "accelerate": "0.29.3",
    "peft": "0.11.1",
}


def parse_version_tuple(version: str) -> tuple[int, ...]:
    parts = re.findall(r"\d+", version)
    return tuple(int(part) for part in parts[:3]) if parts else (0,)


def installed_version(package_name: str) -> str | None:
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def package_is_usable(package_name: str, minimum_version: str) -> bool:
    version = installed_version(package_name)
    if version is None:
        return False
    return parse_version_tuple(version) >= parse_version_tuple(minimum_version)


def offline_wheel_dirs() -> list[str]:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return []
    return sorted({str(path.parent) for path in input_root.rglob("*.whl")})


def run_pip_install(command_args: list[str]) -> None:
    print("Running:", " ".join(command_args))
    subprocess.check_call(command_args)


def install_requirement_offline(package_name: str, requirement: str) -> None:
    wheel_dirs = offline_wheel_dirs()
    if not wheel_dirs:
        raise RuntimeError(
            f"{package_name} is missing or too old, and no offline wheel files were found under "
            "/kaggle/input. Attach a Kaggle Dataset containing compatible wheels for "
            "peft, accelerate, transformers, and their dependencies."
        )

    base_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
    ]

    offline_command = base_command + ["--no-index"]
    for wheel_dir in wheel_dirs:
        offline_command.extend(["--find-links", wheel_dir])
    offline_command.append(requirement)

    print(f"Installing {package_name} from attached offline wheels...")
    try:
        run_pip_install(offline_command)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f"Offline wheel install failed for {package_name}. Make sure the attached wheel "
            f"dataset includes {requirement} plus all required dependencies. Original error: {exc}"
        ) from exc


def ensure_dependencies() -> None:
    for package_name, minimum_version in MIN_DEPENDENCIES.items():
        version = installed_version(package_name)
        if package_is_usable(package_name, minimum_version):
            print(f"{package_name} {version} is available.")
            continue

        requirement = f"{package_name}>={minimum_version}"
        if package_name == "transformers":
            requirement = f"{package_name}>={minimum_version},<5.0.0"

        if version is None:
            print(f"{package_name} is not installed; installing {requirement} from offline wheels.")
        else:
            print(f"{package_name} {version} is too old; installing {requirement} from offline wheels.")
        install_requirement_offline(package_name, requirement)


ensure_dependencies()


import numpy as np
import pandas as pd
import torch
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupKFold
from torch import nn
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


# ---------------------------------------------------------------------------


## Configuration


In [ ]:
# ---------------------------------------------------------------------------

OFFLINE_MODEL_PATHS = [
    "/kaggle/input/modernbert/transformers/large/2",
    "/kaggle/input/modernbert/Transformers/large/2",
    "/kaggle/input/answer-ai-modernbert/transformers/large/2",
    "/kaggle/input/answer-ai-modernbert/Transformers/large/2",
    "/kaggle/input/modernbert-large",
    "/kaggle/input/answerdotai-modernbert-large",
    "/kaggle/input/modernbert-large-offline",
]

COMPETITION_INPUT_DIR = Path("/kaggle/input/llm-classification-finetuning")
LOCAL_INPUT_DIR = Path.cwd() / "data" / "raw"
DEFAULT_OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]
NUM_LABELS = len(LABEL_COLUMNS)
SWAP_LABEL_MAP = np.array([1, 0, 2], dtype=np.int64)


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


# Full-run defaults. Keep LLM_TRAIN_ROW_LIMIT unset/0 for the real submission.
N_FOLDS = env_int("LLM_N_FOLDS", 3)
RANDOM_STATE = env_int("LLM_RANDOM_STATE", 42)
MAX_LENGTH_REQUESTED = env_int("LLM_MAX_LENGTH", 2048)
NUM_EPOCHS = env_int("LLM_NUM_EPOCHS", 1)
TRAIN_BATCH_SIZE = env_int("LLM_TRAIN_BATCH_SIZE", 1)
INFER_BATCH_SIZE = env_int("LLM_INFER_BATCH_SIZE", 2)
GRADIENT_ACCUMULATION_STEPS = env_int("LLM_GRADIENT_ACCUMULATION_STEPS", 16)
LEARNING_RATE = env_float("LLM_LEARNING_RATE", 2e-4)
WEIGHT_DECAY = env_float("LLM_WEIGHT_DECAY", 0.01)
WARMUP_RATIO = env_float("LLM_WARMUP_RATIO", 0.08)
LABEL_SMOOTHING = env_float("LLM_LABEL_SMOOTHING", 0.0)
MAX_GRAD_NORM = env_float("LLM_MAX_GRAD_NORM", 1.0)
TRAIN_ROW_LIMIT = env_int("LLM_TRAIN_ROW_LIMIT", 0)
NUM_WORKERS = env_int("LLM_NUM_WORKERS", 2)
USE_FP16 = env_bool("LLM_USE_FP16", True)
USE_GRADIENT_CHECKPOINTING = env_bool("LLM_USE_GRADIENT_CHECKPOINTING", True)

LORA_RANK = env_int("LLM_LORA_RANK", 16)
LORA_ALPHA = env_int("LLM_LORA_ALPHA", 32)
LORA_DROPOUT = env_float("LLM_LORA_DROPOUT", 0.1)
LORA_TARGET_MODULES = [
    item.strip()
    for item in os.environ.get("LLM_LORA_TARGET_MODULES", "Wqkv,Wi,Wo").split(",")
    if item.strip()
]
MODULES_TO_SAVE = [
    item.strip()
    for item in os.environ.get("LLM_MODULES_TO_SAVE", "head,classifier").split(",")
    if item.strip()
]
CLASSIFIER_POOLING = os.environ.get("LLM_CLASSIFIER_POOLING", "mean").strip().lower()

PROMPT_SHARE = env_float("LLM_PROMPT_SHARE", 0.18)
RESPONSE_A_SHARE = env_float("LLM_RESPONSE_A_SHARE", 0.41)
RESPONSE_B_SHARE = env_float("LLM_RESPONSE_B_SHARE", 0.41)
PROMPT_HEAD_RATIO = env_float("LLM_PROMPT_HEAD_RATIO", 0.80)
RESPONSE_HEAD_RATIO = env_float("LLM_RESPONSE_HEAD_RATIO", 0.72)
PROBABILITY_EPS = env_float("LLM_PROBABILITY_EPS", 1e-7)

OUTPUT_DIR = Path(os.environ.get("LLM_OUTPUT_DIR", str(DEFAULT_OUTPUT_DIR)))


@dataclass(frozen=True)
class EncodedRow:
    prompt_ids: list[int]
    response_a_ids: list[int]
    response_b_ids: list[int]


# ---------------------------------------------------------------------------


## General Helpers


In [ ]:
# ---------------------------------------------------------------------------


def print_rule(title: str) -> None:
    print()
    print("=" * 90)
    print(title)
    print("=" * 90)


def format_seconds(seconds: float) -> str:
    minutes, sec = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h {minutes}m {sec}s"


def resolve_model_name() -> str:
    env_model_path = os.environ.get("LLM_MODEL_PATH")
    if env_model_path:
        path = Path(env_model_path)
        if path.exists():
            print(f"Using offline ModernBERT weights from LLM_MODEL_PATH: {path}")
            return str(path)
        raise FileNotFoundError(f"LLM_MODEL_PATH was set but does not exist: {path}")

    for path in OFFLINE_MODEL_PATHS:
        if Path(path).exists():
            print(f"Found offline ModernBERT weights at: {path}")
            return path

    if Path("/kaggle/input").exists():
        for config_path in sorted(Path("/kaggle/input").rglob("config.json")):
            try:
                payload = json.loads(config_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            model_type = str(payload.get("model_type", "")).lower()
            architectures = " ".join(str(item) for item in payload.get("architectures", [])).lower()
            if "modernbert" in model_type or "modernbert" in architectures:
                print(f"Discovered offline ModernBERT weights at: {config_path.parent}")
                return str(config_path.parent)

    raise FileNotFoundError(
        "No offline ModernBERT weights were found under /kaggle/input. Attach the Kaggle "
        "ModernBERT model/dataset, or set LLM_MODEL_PATH to the attached model directory. "
        "No network model loading is attempted."
    )


MODEL_NAME = resolve_model_name()


def seed_everything(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def seed_worker(worker_id: int) -> None:
    worker_seed = (RANDOM_STATE + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def find_data_dir() -> Path:
    if COMPETITION_INPUT_DIR.exists():
        return COMPETITION_INPUT_DIR
    if Path("/kaggle/input").exists():
        matches = sorted(Path("/kaggle/input").rglob("train.csv"))
        if matches:
            return matches[0].parent
    return LOCAL_INPUT_DIR


def load_model_config():
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    if hasattr(config, "classifier_pooling"):
        if CLASSIFIER_POOLING not in {"cls", "mean"}:
            raise ValueError("LLM_CLASSIFIER_POOLING must be either 'cls' or 'mean'.")
        config.classifier_pooling = CLASSIFIER_POOLING
    if hasattr(config, "classifier_dropout"):
        config.classifier_dropout = LORA_DROPOUT
    if hasattr(config, "reference_compile"):
        config.reference_compile = False
    return config


def effective_max_length(config) -> int:
    model_max_length = getattr(config, "max_position_embeddings", None)
    if model_max_length and MAX_LENGTH_REQUESTED > model_max_length:
        print(
            f"Requested max length {MAX_LENGTH_REQUESTED} exceeds model limit "
            f"{model_max_length}; using {model_max_length}."
        )
        return int(model_max_length)
    return MAX_LENGTH_REQUESTED


def validate_input_frames(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    train_required = ["id", "prompt", "response_a", "response_b", *LABEL_COLUMNS]
    test_required = ["id", "prompt", "response_a", "response_b"]
    missing_train = [col for col in train_required if col not in train_df.columns]
    missing_test = [col for col in test_required if col not in test_df.columns]
    if missing_train:
        raise ValueError(f"train.csv is missing required columns: {missing_train}")
    if missing_test:
        raise ValueError(f"test.csv is missing required columns: {missing_test}")

    label_matrix = train_df[LABEL_COLUMNS].astype(float).values
    row_sums = label_matrix.sum(axis=1)
    invalid_count = int((row_sums != 1).sum())
    if invalid_count:
        raise ValueError(f"Found {invalid_count} train rows without exactly one winning label.")


def build_labels(df: pd.DataFrame) -> np.ndarray:
    return df[LABEL_COLUMNS].astype(float).values.argmax(axis=1).astype(np.int64)


def build_groups(df: pd.DataFrame) -> np.ndarray:
    group_ids, _ = pd.factorize(df["prompt"].fillna("").astype(str), sort=False)
    return group_ids.astype(np.int64)


def limit_training_rows(df: pd.DataFrame) -> pd.DataFrame:
    if TRAIN_ROW_LIMIT <= 0 or TRAIN_ROW_LIMIT >= len(df):
        return df
    label_ids = build_labels(df)
    limited_indices = (
        pd.DataFrame({"row_index": np.arange(len(df)), "label": label_ids})
        .groupby("label", group_keys=False)
        .sample(frac=TRAIN_ROW_LIMIT / len(df), random_state=RANDOM_STATE)
        .head(TRAIN_ROW_LIMIT)["row_index"]
        .to_numpy()
    )
    if len(limited_indices) < TRAIN_ROW_LIMIT:
        remaining = TRAIN_ROW_LIMIT - len(limited_indices)
        pool = np.setdiff1d(np.arange(len(df)), limited_indices, assume_unique=False)
        extra = np.random.default_rng(RANDOM_STATE).choice(pool, size=remaining, replace=False)
        limited_indices = np.concatenate([limited_indices, extra])
    limited_df = df.iloc[limited_indices].sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    if len(limited_df) < N_FOLDS * NUM_LABELS:
        raise ValueError(f"LLM_TRAIN_ROW_LIMIT={TRAIN_ROW_LIMIT} is too small for {N_FOLDS} folds.")
    print(f"Using a limited training set of {len(limited_df):,} rows.")
    return limited_df


def normalize_probabilities(probs: np.ndarray) -> np.ndarray:
    probs = np.asarray(probs, dtype=np.float64)
    probs = np.clip(probs, PROBABILITY_EPS, 1.0 - PROBABILITY_EPS)
    row_sums = probs.sum(axis=1, keepdims=True)
    return (probs / row_sums).astype(np.float32)


def validate_submission(submission_df: pd.DataFrame, expected_rows: int) -> None:
    expected_columns = ["id", *LABEL_COLUMNS]
    if list(submission_df.columns) != expected_columns:
        raise ValueError(f"Submission columns must be {expected_columns}; got {list(submission_df.columns)}")
    if len(submission_df) != expected_rows:
        raise ValueError(f"Submission row count mismatch: {len(submission_df)} != {expected_rows}")
    probs = submission_df[LABEL_COLUMNS].values
    if not np.isfinite(probs).all():
        raise ValueError("Submission probabilities contain NaN or infinite values.")
    if (probs < 0).any() or (probs > 1).any():
        raise ValueError("Submission probabilities must stay within [0, 1].")
    max_sum_error = float(np.abs(probs.sum(axis=1) - 1.0).max())
    if max_sum_error > 1e-5:
        raise ValueError(f"Submission probability rows do not sum to 1. Max error: {max_sum_error}")


# ---------------------------------------------------------------------------


## Tokenization And Truncation


In [ ]:
# ---------------------------------------------------------------------------


def truncate_head_tail(token_ids: list[int], budget: int, head_ratio: float) -> list[int]:
    if budget <= 0:
        return []
    if len(token_ids) <= budget:
        return token_ids
    head_count = max(1, int(round(budget * head_ratio)))
    head_count = min(head_count, budget - 1)
    tail_count = budget - head_count
    if tail_count <= 0:
        return token_ids[:budget]
    return token_ids[:head_count] + token_ids[-tail_count:]


def allocate_budgets(lengths: list[int], total_budget: int, shares: list[float]) -> list[int]:
    budgets = [min(length, int(total_budget * share)) for length, share in zip(lengths, shares)]
    assigned = sum(budgets)
    remaining = max(0, total_budget - assigned)
    while remaining > 0:
        candidate = max(range(len(lengths)), key=lambda idx: (lengths[idx] - budgets[idx], lengths[idx]))
        if budgets[candidate] >= lengths[candidate]:
            break
        budgets[candidate] += 1
        remaining -= 1
    return budgets


def build_model_inputs(
    encoded_row: EncodedRow,
    tokenizer,
    max_length: int,
    swap_responses: bool,
) -> tuple[list[int], list[int]]:
    prompt_ids = encoded_row.prompt_ids
    response_a_ids = encoded_row.response_b_ids if swap_responses else encoded_row.response_a_ids
    response_b_ids = encoded_row.response_a_ids if swap_responses else encoded_row.response_b_ids

    content_budget = max_length - 4
    budgets = allocate_budgets(
        lengths=[len(prompt_ids), len(response_a_ids), len(response_b_ids)],
        total_budget=content_budget,
        shares=[PROMPT_SHARE, RESPONSE_A_SHARE, RESPONSE_B_SHARE],
    )

    prompt_final = truncate_head_tail(prompt_ids, budgets[0], PROMPT_HEAD_RATIO)
    response_a_final = truncate_head_tail(response_a_ids, budgets[1], RESPONSE_HEAD_RATIO)
    response_b_final = truncate_head_tail(response_b_ids, budgets[2], RESPONSE_HEAD_RATIO)

    input_ids = [
        tokenizer.cls_token_id,
        *prompt_final,
        tokenizer.sep_token_id,
        *response_a_final,
        tokenizer.sep_token_id,
        *response_b_final,
        tokenizer.sep_token_id,
    ]
    return input_ids, [1] * len(input_ids)


def encode_texts(tokenizer, texts: list[str], batch_size: int = 256) -> list[list[int]]:
    all_ids: list[list[int]] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        tokenized = tokenizer(batch, add_special_tokens=False, truncation=False, verbose=False)
        all_ids.extend(tokenized["input_ids"])
    return all_ids


def pretokenize_dataframe(df: pd.DataFrame, tokenizer) -> list[EncodedRow]:
    prompt_ids = encode_texts(tokenizer, ("Prompt:\n" + df["prompt"].fillna("").astype(str)).tolist())
    response_a_ids = encode_texts(tokenizer, ("Response A:\n" + df["response_a"].fillna("").astype(str)).tolist())
    response_b_ids = encode_texts(tokenizer, ("Response B:\n" + df["response_b"].fillna("").astype(str)).tolist())
    return [EncodedRow(p, a, b) for p, a, b in zip(prompt_ids, response_a_ids, response_b_ids)]


# ---------------------------------------------------------------------------


## Dataset And Collation


In [ ]:
# ---------------------------------------------------------------------------


class PreferenceDataset:
    def __init__(
        self,
        encoded_rows: list[EncodedRow],
        row_indices: np.ndarray,
        labels: np.ndarray | None,
        tokenizer,
        max_length: int,
        swap_flags: np.ndarray | None = None,
        swap_labels: bool = False,
    ) -> None:
        self.encoded_rows = encoded_rows
        self.row_indices = row_indices.astype(np.int64)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.swap_flags = (
            swap_flags.astype(bool) if swap_flags is not None else np.zeros(len(self.row_indices), dtype=bool)
        )
        self.swap_labels = swap_labels

    def __len__(self) -> int:
        return len(self.row_indices)

    def __getitem__(self, index: int) -> dict[str, list[int] | int]:
        row_index = int(self.row_indices[index])
        swap = bool(self.swap_flags[index])
        input_ids, attention_mask = build_model_inputs(
            self.encoded_rows[row_index],
            self.tokenizer,
            self.max_length,
            swap,
        )
        item: dict[str, list[int] | int] = {"input_ids": input_ids, "attention_mask": attention_mask}
        if self.labels is not None:
            label = int(self.labels[row_index])
            if self.swap_labels and swap:
                label = int(SWAP_LABEL_MAP[label])
            item["labels"] = label
        return item


def build_collate_fn(tokenizer):
    pad_id = tokenizer.pad_token_id

    def collate_fn(batch):
        max_len = max(len(item["input_ids"]) for item in batch)
        input_ids, attention_mask, labels = [], [], []
        for item in batch:
            pad_width = max_len - len(item["input_ids"])
            input_ids.append(item["input_ids"] + [pad_id] * pad_width)
            attention_mask.append(item["attention_mask"] + [0] * pad_width)
            if "labels" in item:
                labels.append(item["labels"])
        payload = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }
        if labels:
            payload["labels"] = torch.tensor(labels, dtype=torch.long)
        return payload

    return collate_fn


def build_train_indices_with_swap(indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    original_indices = indices.astype(np.int64)
    swapped_indices = indices.astype(np.int64)
    merged_indices = np.concatenate([original_indices, swapped_indices])
    swap_flags = np.concatenate(
        [
            np.zeros(len(original_indices), dtype=bool),
            np.ones(len(swapped_indices), dtype=bool),
        ]
    )
    return merged_indices, swap_flags


# ---------------------------------------------------------------------------


## Modeling, Training, And Inference


In [ ]:
# ---------------------------------------------------------------------------


def build_model_with_lora(config):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=MODULES_TO_SAVE,
        bias="none",
    )
    model = get_peft_model(model, lora_config)

    if USE_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            model.gradient_checkpointing_enable()
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()

    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()

    return model


def build_optimizer(model):
    trainable_params = [param for param in model.parameters() if param.requires_grad]
    if not trainable_params:
        raise RuntimeError("No trainable parameters found after applying LoRA.")
    return torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


@np.errstate(over="ignore")
def softmax_np(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return normalize_probabilities(exp_scores / exp_scores.sum(axis=1, keepdims=True))


def autocast_context(device):
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", enabled=USE_FP16 and device.type == "cuda")
    return torch.cuda.amp.autocast(enabled=USE_FP16 and device.type == "cuda")


def predict_probabilities(model, loader, device) -> np.ndarray:
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            with autocast_context(device):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            all_logits.append(logits.detach().cpu().to(torch.float32).numpy())
    return softmax_np(np.vstack(all_logits))


def predict_with_swap_tta(model, loader_original, loader_swapped, device) -> np.ndarray:
    original_probs = predict_probabilities(model, loader_original, device)
    swapped_probs = predict_probabilities(model, loader_swapped, device)
    swapped_back = swapped_probs[:, SWAP_LABEL_MAP]
    return normalize_probabilities(0.5 * (original_probs + swapped_back))


def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler, device, epoch_idx: int) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    for step, batch in enumerate(loader, start=1):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        targets = batch["labels"].to(device, non_blocking=True)

        with autocast_context(device):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, targets)
            scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(scaled_loss).backward()
        running_loss += float(loss.detach().cpu())

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            old_scale = scaler.get_scale() if scaler.is_enabled() else None
            scaler.step(optimizer)
            scaler.update()
            new_scale = scaler.get_scale() if scaler.is_enabled() else None
            optimizer.zero_grad(set_to_none=True)
            if old_scale is None or new_scale >= old_scale:
                scheduler.step()

        if step % 200 == 0 or step == len(loader):
            avg_loss = running_loss / step
            print(f"Epoch {epoch_idx} | step {step:,}/{len(loader):,} | avg_loss={avg_loss:.5f}")

    return running_loss / max(1, len(loader))


def save_best_adapter(model, tokenizer, save_dir: Path, metrics: dict) -> None:
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    (save_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")


def clear_torch_memory() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


# ---------------------------------------------------------------------------


## Run Training And Write Submission


In [ ]:
# ---------------------------------------------------------------------------


def main() -> int:
    run_start = time.time()
    seed_everything()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print("WARNING: CUDA is not available. A full ModernBERT-large run will be extremely slow.")

    data_dir = find_data_dir()
    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")
    validate_input_frames(train_df, test_df)
    train_df = limit_training_rows(train_df)

    config = load_model_config()
    max_length = effective_max_length(config)

    print_rule("Run configuration")
    print(f"Data dir:                      {data_dir}")
    print(f"Output dir:                    {OUTPUT_DIR}")
    print(f"Model:                         {MODEL_NAME}")
    print(f"Device:                        {device}")
    print(f"Train rows:                    {len(train_df):,}")
    print(f"Test rows:                     {len(test_df):,}")
    print(f"Folds:                         {N_FOLDS}")
    print(f"Epochs per fold:               {NUM_EPOCHS}")
    print(f"Max length requested/effective: {MAX_LENGTH_REQUESTED}/{max_length}")
    print(f"Classifier pooling:            {getattr(config, 'classifier_pooling', 'unknown')}")
    print(f"Train batch size:              {TRAIN_BATCH_SIZE}")
    print(f"Inference batch size:          {INFER_BATCH_SIZE}")
    print(f"Gradient accumulation:         {GRADIENT_ACCUMULATION_STEPS}")
    print(f"Learning rate:                 {LEARNING_RATE}")
    print(f"LoRA rank/alpha/dropout:       {LORA_RANK}/{LORA_ALPHA}/{LORA_DROPOUT}")
    print(f"LoRA target modules:           {LORA_TARGET_MODULES}")
    print(f"Modules to save:               {MODULES_TO_SAVE}")
    print(f"FP16 autocast:                 {USE_FP16}")
    print(f"Gradient checkpointing:        {USE_GRADIENT_CHECKPOINTING}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token

    print_rule("Pretokenizing train/test fields")
    encoded_train = pretokenize_dataframe(train_df, tokenizer)
    encoded_test = pretokenize_dataframe(test_df, tokenizer)

    labels = build_labels(train_df)
    groups = build_groups(train_df)
    unique_groups = np.unique(groups)
    if len(unique_groups) < N_FOLDS:
        raise ValueError(f"Need at least {N_FOLDS} unique prompt groups; found {len(unique_groups)}.")

    collate_fn = build_collate_fn(tokenizer)
    splitter = GroupKFold(n_splits=N_FOLDS)
    torch_generator = torch.Generator()
    torch_generator.manual_seed(RANDOM_STATE)

    oof_probs = np.zeros((len(train_df), NUM_LABELS), dtype=np.float32)
    test_probs_accum = np.zeros((len(test_df), NUM_LABELS), dtype=np.float32)

    for fold_idx, (train_idx, val_idx) in enumerate(splitter.split(train_df, labels, groups=groups), start=1):
        fold_start = time.time()
        print_rule(f"Fold {fold_idx}/{N_FOLDS}")
        print(f"Train rows before swap: {len(train_idx):,}")
        print(f"Validation rows:        {len(val_idx):,}")

        train_indices_aug, train_swap_flags = build_train_indices_with_swap(train_idx)
        val_indices = val_idx.astype(np.int64)
        test_indices = np.arange(len(test_df), dtype=np.int64)

        train_dataset = PreferenceDataset(
            encoded_rows=encoded_train,
            row_indices=train_indices_aug,
            labels=labels,
            tokenizer=tokenizer,
            max_length=max_length,
            swap_flags=train_swap_flags,
            swap_labels=True,
        )
        val_dataset_orig = PreferenceDataset(
            encoded_rows=encoded_train,
            row_indices=val_indices,
            labels=labels,
            tokenizer=tokenizer,
            max_length=max_length,
        )
        val_dataset_swap = PreferenceDataset(
            encoded_rows=encoded_train,
            row_indices=val_indices,
            labels=labels,
            tokenizer=tokenizer,
            max_length=max_length,
            swap_flags=np.ones(len(val_indices), dtype=bool),
        )
        test_dataset_orig = PreferenceDataset(
            encoded_rows=encoded_test,
            row_indices=test_indices,
            labels=None,
            tokenizer=tokenizer,
            max_length=max_length,
        )
        test_dataset_swap = PreferenceDataset(
            encoded_rows=encoded_test,
            row_indices=test_indices,
            labels=None,
            tokenizer=tokenizer,
            max_length=max_length,
            swap_flags=np.ones(len(test_indices), dtype=bool),
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=TRAIN_BATCH_SIZE,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
            worker_init_fn=seed_worker,
            generator=torch_generator,
        )
        val_loader_orig = DataLoader(
            val_dataset_orig,
            batch_size=INFER_BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        val_loader_swap = DataLoader(
            val_dataset_swap,
            batch_size=INFER_BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        test_loader_orig = DataLoader(
            test_dataset_orig,
            batch_size=INFER_BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        test_loader_swap = DataLoader(
            test_dataset_swap,
            batch_size=INFER_BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )

        model = build_model_with_lora(config).to(device)
        optimizer = build_optimizer(model)
        update_steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
        total_update_steps = max(1, update_steps_per_epoch * NUM_EPOCHS)
        warmup_steps = int(total_update_steps * WARMUP_RATIO)
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
        if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
            scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16 and device.type == "cuda")
        else:
            scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16 and device.type == "cuda")

        best_val_loss = float("inf")
        best_adapter_state = None
        best_epoch = 0

        for epoch_idx in range(1, NUM_EPOCHS + 1):
            epoch_start = time.time()
            train_loss = train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                scheduler=scheduler,
                criterion=criterion,
                scaler=scaler,
                device=device,
                epoch_idx=epoch_idx,
            )

            print("Evaluating validation fold with swap TTA...")
            val_probs_tta = predict_with_swap_tta(model, val_loader_orig, val_loader_swap, device)
            val_loss = log_loss(labels[val_idx], val_probs_tta, labels=[0, 1, 2])
            print(
                f"Fold {fold_idx} | epoch {epoch_idx} "
                f"| train_loss={train_loss:.5f} "
                f"| val_log_loss={val_loss:.5f} "
                f"| epoch_runtime={format_seconds(time.time() - epoch_start)}"
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch_idx
                best_adapter_state = {
                    key: value.detach().cpu().clone()
                    for key, value in get_peft_model_state_dict(model).items()
                }
                oof_probs[val_idx] = val_probs_tta
                save_best_adapter(
                    model=model,
                    tokenizer=tokenizer,
                    save_dir=OUTPUT_DIR / f"fold_{fold_idx}",
                    metrics={
                        "fold": fold_idx,
                        "best_epoch": best_epoch,
                        "best_val_log_loss": best_val_loss,
                        "max_length": max_length,
                        "model_name": MODEL_NAME,
                    },
                )

        if best_adapter_state is None:
            raise RuntimeError(f"No best adapter state captured for fold {fold_idx}.")

        set_peft_model_state_dict(model, best_adapter_state)
        model.to(device)

        print(f"Predicting test set with best fold {fold_idx} adapter from epoch {best_epoch}...")
        test_probs_fold = predict_with_swap_tta(model, test_loader_orig, test_loader_swap, device)
        test_probs_accum += test_probs_fold.astype(np.float32)

        print(f"Best fold {fold_idx} validation log loss: {best_val_loss:.5f}")
        print(f"Fold {fold_idx} runtime: {format_seconds(time.time() - fold_start)}")

        del model, optimizer, scheduler, train_loader, val_loader_orig, val_loader_swap
        del test_loader_orig, test_loader_swap
        clear_torch_memory()

    oof_probs = normalize_probabilities(oof_probs)
    overall_log_loss = log_loss(labels, oof_probs, labels=[0, 1, 2])
    print_rule("Cross-validation summary")
    print(f"OOF log loss across {N_FOLDS} folds: {overall_log_loss:.5f}")

    test_probs = normalize_probabilities(test_probs_accum / N_FOLDS)
    submission_df = pd.DataFrame(
        {
            "id": test_df["id"],
            LABEL_COLUMNS[0]: test_probs[:, 0],
            LABEL_COLUMNS[1]: test_probs[:, 1],
            LABEL_COLUMNS[2]: test_probs[:, 2],
        }
    )
    validate_submission(submission_df, expected_rows=len(test_df))

    submission_path = OUTPUT_DIR / "submission.csv"
    submission_df.to_csv(submission_path, index=False)

    oof_path = OUTPUT_DIR / "oof_predictions.csv"
    pd.DataFrame(
        {
            "id": train_df["id"],
            LABEL_COLUMNS[0]: oof_probs[:, 0],
            LABEL_COLUMNS[1]: oof_probs[:, 1],
            LABEL_COLUMNS[2]: oof_probs[:, 2],
        }
    ).to_csv(oof_path, index=False)

    metrics = {
        "oof_log_loss": float(overall_log_loss),
        "n_folds": N_FOLDS,
        "epochs": NUM_EPOCHS,
        "max_length": max_length,
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "model_name": MODEL_NAME,
        "runtime_seconds": int(time.time() - run_start),
    }
    (OUTPUT_DIR / "run_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    print_rule("Submission file written")
    print(f"Submission path: {submission_path}")
    print(f"OOF path:        {oof_path}")
    print(f"Metrics path:    {OUTPUT_DIR / 'run_metrics.json'}")
    print(submission_df.head())
    print(f"Total runtime:   {format_seconds(time.time() - run_start)}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Expected Outputs

After a successful run, Kaggle should contain `submission.csv`, `oof_predictions.csv`, `run_metrics.json`, and one saved LoRA adapter directory per fold under `/kaggle/working`.
